# Đánh giá Hybrid và Reranker trên Colab

Phần Người 1 Team 2: VLRA-54/55 và bằng chứng cho VLRA-58.
Model/index nằm trên Colab; máy cá nhân chỉ mở trình duyệt.
Không cần API key, Ollama, Java, Docker hoặc mount Drive.

Chọn **Runtime → Change runtime type → GPU**, rồi chạy từ trên xuống.
[Colab miễn phí không bảo đảm GPU hoặc thời lượng phiên](https://research.google.com/colaboratory/faq.html#resource-limits).
Notebook không mua tài nguyên.

Mặc định khảo sát bộ **110 câu** của Team 1: 40 câu đã review theo nhãn
nguồn, 70 câu còn draft. Không đổi nhãn draft thành đã review hoặc coi đây
là nghiệm thu Dev/Test. Để tái lập bộ 40 câu cũ, đổi GOLDEN ở ô 3.

Đo Hybrid top 20, BGE gốc và BGE với `truncate=True` của Người 2 trên
**cùng pool từng câu**, giữ RRF k=60, revision model và batch 2. Cả hai
chính sách rerank chạy strict, không fallback; không sửa passage gốc.


## 1. Lấy code và cài thư viện


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Chỉ dùng PyTorch; tránh framework khác cấp trước VRAM của Colab.
os.environ["HF_HOME"] = "/content/hf-cache"
os.environ["TORCH_HOME"] = "/content/torch-cache"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

REPO_URL = "https://github.com/Banh-Van-Hiep/vietnamese-legal-rag.git"
BRANCH = "feature/team2-rag"
REPO = Path("/content/vietnamese-legal-rag")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    # Giữ checkout đã có; không pull hoặc xóa thay đổi khi chạy lại ô.
    origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    if origin != REPO_URL:
        raise RuntimeError("Thư mục hiện có không thuộc repo này; hãy dùng runtime mới.")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if subprocess.check_output(["git", "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Checkout có thay đổi chưa commit; hãy kiểm tra trước khi đo.")
print("Code:", HEAD)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[test,retrieval,dense,reranker]"], check=True)


## 2. Kiểm tra GPU


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Chưa có CUDA GPU. Chọn GPU trong Runtime → Change runtime type, rồi kết nối lại.")
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name())
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)


## 3. Chọn dữ liệu và ghim revision model

Giữ `MODE = "exploratory"` cho bộ 110 câu còn draft. Muốn tái lập khảo sát
40 câu, đặt GOLDEN thành `team2_rag/data/golden/golden_phase1_30cai.json`.

Chỉ đổi MODE thành `dev` khi có file được review/đóng băng, có `split: "dev"`
và `relevant_unit_groups`. Không dùng Test để chọn cấu hình. Notebook giữ
nguyên nhãn nguồn; hash Golden được ghi và bản sao được đóng gói trong ZIP.

Revision mặc định giữ cùng hai model của lượt 31/10 để đối chiếu.


In [ ]:
import hashlib
import json
from collections import Counter
from datetime import datetime, timezone
from huggingface_hub import model_info
from scripts.eval_reranker import validate_dataset

MODE = "exploratory"
GOLDEN = REPO / "team1_data/golden/golden_phase1.json"
CHUNKS = REPO / "data/doanh-nghiep/processed/chunks.jsonl"
EMBEDDING_MODEL = "AITeamVN/Vietnamese_Embedding"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
EMBEDDING_REVISION = "dea33aa1ab339f38d66ae0a40e6c40e0a9249568"
RERANKER_REVISION = "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e"

if MODE not in ("exploratory", "dev"):
    raise ValueError("MODE phải là exploratory hoặc dev.")
cases = json.loads(GOLDEN.read_text(encoding="utf-8"))
chunks = [json.loads(line) for line in CHUNKS.read_text(encoding="utf-8").splitlines() if line.strip()]
corpus = {row["chunk_id"]: row for row in chunks}
if len(corpus) != len(chunks):
    raise ValueError("Corpus có chunk_id trùng.")
validate_dataset(cases, corpus, exploratory=MODE == "exploratory")
EMBEDDING_REVISION = model_info(EMBEDDING_MODEL, revision=EMBEDDING_REVISION or "main").sha
RERANKER_REVISION = model_info(RERANKER_MODEL, revision=RERANKER_REVISION or "main").sha
CORPUS_HASH = hashlib.sha256(CHUNKS.read_bytes()).hexdigest()
VERSION = f"colab-{CORPUS_HASH[:12]}-{EMBEDDING_REVISION[:8]}"
RUN = Path("/content/legal-rag-runs") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
EVIDENCE = RUN / "evidence"
EVIDENCE.mkdir(parents=True, exist_ok=False)
BM25_INDEX, DENSE_INDEX, QDRANT = [RUN / name for name in ("bm25", "dense", "qdrant")]
settings = dict(mode=MODE, head=HEAD, corpus_sha256=CORPUS_HASH,
                golden_sha256=hashlib.sha256(GOLDEN.read_bytes()).hexdigest(),
                index_version=VERSION, embedding_model=EMBEDDING_MODEL,
                embedding_revision=EMBEDDING_REVISION, reranker_revision=RERANKER_REVISION,
                compare_truncation=True, strict=True,
                review_status=dict(Counter(c.get("review_status", "unspecified") for c in cases)),
                device=DEVICE, gpu=torch.cuda.get_device_name(), torch=torch.__version__, cuda=torch.version.cuda)
(EVIDENCE / "run-settings.json").write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding="utf-8")
(EVIDENCE / "golden.json").write_bytes(GOLDEN.read_bytes())
(EVIDENCE / "packages.txt").write_bytes(subprocess.check_output([sys.executable, "-m", "pip", "freeze"]))
print("Số câu:", len(cases), dict(Counter(c["expected_status"] for c in cases)))
print("Corpus:", len(chunks), "chunks; chế độ:", MODE)
print("Revision embedding:", EMBEDDING_REVISION, "reranker:", RERANKER_REVISION)
print("Bằng chứng:", EVIDENCE)


## 4. Dựng index trên Colab

Dùng trực tiếp thư viện Team 1, giữ nguyên chunk và thuật toán. Cả hai index
mang cùng version và tập IDs. Đây là index mới để thực nghiệm từ corpus của
commit đã ghi; không mặc định là snapshot MVP trước đây.

Ô này tải embedding model về Colab. Build chạy trong tiến trình riêng, đóng Qdrant và trả toàn bộ VRAM
trước khi runner nạp lại model. Nếu build dở, chạy lại từ ô 3 để tạo thư mục
run mới; không xóa hoặc ghi đè index đang có.


In [ ]:
import shutil

# Tiến trình build kết thúc sẽ trả toàn bộ VRAM trước khi runner nạp hai model.
BUILD_CODE = r"""
import json
import sys
from pathlib import Path

settings = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
CHUNKS = Path(sys.argv[2])
EVIDENCE = Path(sys.argv[1]).parent
RUN = EVIDENCE.parent
BM25_INDEX, DENSE_INDEX, QDRANT = [RUN / name for name in ("bm25", "dense", "qdrant")]
VERSION = settings["index_version"]
EMBEDDING_MODEL, EMBEDDING_REVISION = settings["embedding_model"], settings["embedding_revision"]
DEVICE = settings["device"]
corpus = {json.loads(line)["chunk_id"] for line in CHUNKS.read_text(encoding="utf-8").splitlines() if line.strip()}

import shutil
import time
from qdrant_client import QdrantClient
from team1_data.retrieval import BM25Retriever, DenseRetriever, SentenceTransformerEmbedder

if any(path.exists() for path in (BM25_INDEX, DENSE_INDEX, QDRANT)):
    raise RuntimeError("Run này đã có index. Chạy lại từ ô 3 để tạo run mới.")
start = time.perf_counter()
sparse = BM25Retriever.from_jsonl(CHUNKS)
if set(sparse.chunk_ids) != set(corpus):
    raise ValueError("Bộ lọc hiệu lực làm thay đổi tập chunk; cần đối chiếu snapshot trước khi đo.")
sparse.save(BM25_INDEX, index_version=VERSION)
embedder = SentenceTransformerEmbedder(name=EMBEDDING_MODEL, revision=EMBEDDING_REVISION,
                                       batch_size=8, device=DEVICE)
client = QdrantClient(path=str(QDRANT))
try:
    dense = DenseRetriever.from_jsonl(CHUNKS, embedder=embedder, client=client, index_version=VERSION)
    if set(dense.chunk_ids) != set(sparse.chunk_ids):
        raise ValueError("Hai index không cùng tập chunk IDs.")
    dense.save(DENSE_INDEX)
finally:
    client.close()
for name, path in (("bm25", BM25_INDEX), ("dense", DENSE_INDEX)):
    shutil.copy2(path / "index_manifest.json", EVIDENCE / f"{name}-manifest.json")
(EVIDENCE / "build-time.json").write_text(json.dumps(dict(elapsed_ms=(time.perf_counter() - start) * 1000)))
print("Đã dựng index:", VERSION)
"""
subprocess.run([sys.executable, "-c", BUILD_CODE, str(EVIDENCE / "run-settings.json"), str(CHUNKS)],
               cwd=REPO, check=True)
print("Đã dựng index:", VERSION)


## 5. So sánh bản gốc và bản truncate

Mỗi câu chỉ retrieval một lần. BGE gốc và BGE truncate dùng cùng pool và
weights; chỉ thay chính sách tokenization. Bản gốc ghi lỗi nếu vượt 512 token;
bản truncate dùng `only_second`, giữ câu hỏi và cắt đuôi passage ở đầu vào
model. Nội dung chunk/citation trong output vẫn nguyên vẹn.

Exit 0 là cả hai chính sách chạy đủ; 1 có lỗi ở ít nhất một nhánh; 2 bị
chặn lúc chuẩn bị. Không coi fallback là rerank thành công. So sánh chất
lượng hai chính sách trên giao các câu thành công; giữ riêng số liệu trên
mọi câu thành công của từng nhánh để tránh đổi mẫu số mà không báo.

Latency tổng mỗi nhánh cộng retrieval/kiểm tra đã đo với xử lý riêng nhánh
đó, không gồm thời gian chạy nhánh còn lại. Chưa phải latency LLM/API/E2E.


In [ ]:
from IPython.display import Markdown, display

REPORT = EVIDENCE / f"reranker-{MODE}.json"
command = [sys.executable, "-m", "scripts.eval_reranker",
           "--golden", str(GOLDEN), "--chunks", str(CHUNKS),
           "--bm25-index", str(BM25_INDEX), "--dense-index", str(DENSE_INDEX),
           "--qdrant-path", str(QDRANT), "--device", DEVICE, "--embedding-device", DEVICE,
           "--reranker-revision", RERANKER_REVISION, "--compare-truncation", "--output", str(REPORT)]
if MODE == "exploratory":
    command.append("--exploratory")
# Không raise ngay: giữ quyền đọc và tải báo cáo của run bị lỗi.
result = subprocess.run(command, cwd=REPO)
print("Exit code:", result.returncode)
if REPORT.exists():
    report = json.loads(REPORT.read_text(encoding="utf-8"))
    print("Trạng thái:", report["status"])
    display(Markdown(REPORT.with_suffix(".md").read_text(encoding="utf-8")))
else:
    print("Chưa có báo cáo. Kiểm tra lỗi phía trên rồi chạy lại ô này.")


## 6. Tải bằng chứng về máy

ZIP chứa báo cáo, raw candidates/scores, Golden đầu vào, manifest index,
revision model và danh sách thư viện. Không đóng gói weights hoặc vector
index lớn. Tải ngay khi kết thúc vì filesystem Colab có thể mất khi phiên dừng.

Dùng raw cases để phân tích VLRA-55. Với chế độ exploratory, ghi đúng phạm vi
khảo sát; chưa chốt cấu hình VLRA-58 hoặc tuyên bố đạt benchmark chính thức.


In [ ]:
import base64
from IPython.display import HTML, display

framework_env = {key: os.environ.get(key) for key in ("HF_HOME", "TORCH_HOME", "USE_TF", "USE_FLAX", "XLA_PYTHON_CLIENT_PREALLOCATE")}
(EVIDENCE / "framework-env.json").write_text(json.dumps(framework_env, indent=2), encoding="utf-8")
# Kiểm tra độc lập độ dài các cặp của câu bị lỗi, không nạp lại weights.
failed = [row for row in report.get("cases", []) if row["status"] == "RERANK_ERROR"] if REPORT.exists() else []
if failed:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-reranker-v2-m3", revision=RERANKER_REVISION,
                                             trust_remote_code=False)
    checks = []
    for row in failed:
        candidates = row["candidates"]
        encoded = tokenizer([row["question"]] * len(candidates), [c["content"] for c in candidates],
                            padding=False, truncation=False)
        lengths = [len(ids) for ids in encoded["input_ids"]]
        checks.append(dict(id=row["id"], max_pair_tokens=max(lengths, default=0),
                           oversized=[dict(chunk_id=c["chunk_id"], tokens=n) for c, n in zip(candidates, lengths) if n > 512]))
    (EVIDENCE / "reranker-length-check.json").write_text(json.dumps(checks, indent=2), encoding="utf-8")
archive = shutil.make_archive(str(RUN / "reranker-results"), "zip", root_dir=EVIDENCE)
payload = base64.b64encode(Path(archive).read_bytes()).decode("ascii")
display(HTML(f'<a download="reranker-results.zip" href="data:application/zip;base64,{payload}">Tải ZIP bằng chứng</a>'))
